# 프롬프트별 결과 병합

동일한 모델명과 문항유형의 ao, cot, cove 파일들을 병합합니다.

- ao 파일: 기본키, ques, label, ans → ao_ans
- cot 파일: ans → cot_ans
- cove 파일: final_ans → cove_ans


In [ ]:
import pandas as pd
import os
from pathlib import Path
import re


In [ ]:
# 경로 설정
input_path = Path('output/TFQ23/')
output_path = Path('output/TFQ23/merged/')

# 출력 폴더 생성
output_path.mkdir(exist_ok=True, parents=True)

print(f"입력 폴더: {input_path.absolute()}")
print(f"출력 폴더: {output_path.absolute()}")


In [ ]:
# 해당 input_path 폴더의 모든 xlsx 파일에 idx_num 컬럼을 추가하고, idx_num 컬럼이 전체 컬럼 중 가장 먼저 오도록 설정
# idx_num은 행 번호로, 1~100까지 설정

# input_path의 모든 xlsx 파일 찾기
xlsx_files = list(input_path.glob('*.xlsx'))

print(f"처리할 파일 수: {len(xlsx_files)}")

for file_path in xlsx_files:
    # 임시 파일 제외
    if file_path.name.startswith('~$'):
        continue
    
    try:
        # 엑셀 파일 읽기
        df = pd.read_excel(file_path)
        
        # idx_num 컬럼이 이미 있으면 제거 (중복 방지)
        if 'idx_num' in df.columns:
            df = df.drop(columns=['idx_num'])
        
        # idx_num 컬럼 추가 (1부터 시작하는 행 번호)
        df.insert(0, 'idx_num', range(1, len(df) + 1))
        
        # 파일 저장 (기존 파일 덮어쓰기)
        df.to_excel(file_path, index=False)
        print(f"  ✓ {file_path.name}: {len(df)}행 처리 완료")
        
    except Exception as e:
        print(f"  ✗ {file_path.name}: 오류 발생 - {e}")

print(f"\n모든 파일 처리 완료!")



In [ ]:
# 기본키 정의
def get_primary_keys(question_type):
    """문항 유형에 따라 기본키 컬럼 반환"""
    if question_type == 'MCQ':
        return ['idx']
    elif question_type == 'MAQ':
        return ['idx', 'n_or_p']
    elif question_type == 'RQ':
        return ['idx', 'n_or_p']
    elif question_type == 'TFQ':
        return ['idx', 'n_or_p', 'T_or_F']
    elif question_type == 'TFQ2' or question_type == 'TFQ3':
        return ['idx_num']
    else:
        return ['idx']  # 기본값


In [ ]:
# 파일명 파싱 함수
def parse_filename(filename):
    """파일명에서 모델명, 문항유형, 프롬프트유형 추출"""
    # 임시 파일 제외
    if filename.startswith('~$'):
        return None, None, None
    
    # 패턴 매칭
    # 1. cove 패턴: {모델명}_{문항유형}_cove3_final_ans(_parsed).xlsx
    # [A-Z0-9]+ 로 변경하여 TFQ2, TFQ3 같은 숫자 포함 문항유형도 인식
    pattern_cove = r'^(.+?)_([A-Z0-9]+)_cove3_final_ans(?:_parsed)?\.xlsx$'
    match = re.match(pattern_cove, filename)
    if match:
        model_name = match.group(1)
        question_type = match.group(2)
        prompt_type = 'cove'
        return model_name, question_type, prompt_type
    
    # 2. ao/cot 패턴: {모델명}_{문항유형}_{프롬프트유형}(_parsed).xlsx
    # [A-Z0-9]+ 로 변경하여 TFQ2, TFQ3 같은 숫자 포함 문항유형도 인식
    pattern_normal = r'^(.+?)_([A-Z0-9]+)_(ao|cot)(?:_parsed)?\.xlsx$'
    match = re.match(pattern_normal, filename)
    if match:
        model_name = match.group(1)
        question_type = match.group(2)
        prompt_type = match.group(3)
        return model_name, question_type, prompt_type
    
    return None, None, None


In [ ]:
# 모든 엑셀 파일 수집 및 그룹화
files = list(input_path.glob('*.xlsx'))

# 디버깅: 각 파일명 파싱 결과 확인
print("=== 파일명 파싱 디버깅 ===")
for file_path in files:
    filename = file_path.name
    model_name, question_type, prompt_type = parse_filename(filename)
    print(f"  {filename}")
    print(f"    → 모델: {model_name}, 문항유형: {question_type}, 프롬프트: {prompt_type}")
print()

# 파일 그룹화: {(모델명, 문항유형): {프롬프트유형: 파일경로}}
file_groups = {}

for file_path in files:
    filename = file_path.name
    model_name, question_type, prompt_type = parse_filename(filename)
    
    if model_name and question_type and prompt_type:
        key = (model_name, question_type)
        if key not in file_groups:
            file_groups[key] = {}
        file_groups[key][prompt_type] = file_path

print(f"총 {len(files)}개 파일 중 {sum(len(v) for v in file_groups.values())}개 파일 인식")
print(f"\n그룹화된 조합: {len(file_groups)}개")
for key, prompts in sorted(file_groups.items()):
    model, qtype = key
    prompt_list = ', '.join(sorted(prompts.keys()))
    print(f"  {model}_{qtype}: {prompt_list}")


In [ ]:
# 병합 함수
def merge_prompt_files(model_name, question_type, prompt_files):
    """동일한 모델명, 문항유형의 프롬프트 파일들을 병합"""
    print(f"\n처리 중: {model_name}_{question_type}")
    
    # 기본키 가져오기
    primary_keys = get_primary_keys(question_type)
    print(f"  기본키: {primary_keys}")
    
    merged_df = None
    
    # 1. ao 파일 처리 (기본 데이터)
    if 'ao' in prompt_files:
        ao_file = prompt_files['ao']
        df_ao = pd.read_excel(ao_file)
        print(f"  ao 파일: {len(df_ao)} 행")
        
        # 필요한 컬럼만 선택: 기본키, ques, label, ans
        columns_to_keep = primary_keys + ['ques', 'label', 'ans']
        available_columns = [col for col in columns_to_keep if col in df_ao.columns]
        df_ao = df_ao[available_columns]
        
        # ans를 ao_ans로 이름 변경
        if 'ans' in df_ao.columns:
            df_ao = df_ao.rename(columns={'ans': 'ao_ans'})
        
        merged_df = df_ao.copy()
    else:
        print(f"  경고: ao 파일이 없습니다.")
    
    # 2. cot 파일 처리
    if 'cot' in prompt_files:
        cot_file = prompt_files['cot']
        df_cot = pd.read_excel(cot_file)
        print(f"  cot 파일: {len(df_cot)} 행")
        
        # 기본키와 ans 컬럼만 선택
        columns_to_keep = primary_keys + ['ans']
        available_columns = [col for col in columns_to_keep if col in df_cot.columns]
        df_cot = df_cot[available_columns]
        
        # ans를 cot_ans로 이름 변경
        if 'ans' in df_cot.columns:
            df_cot = df_cot.rename(columns={'ans': 'cot_ans'})
        
        # 병합
        if merged_df is not None:
            merged_df = merged_df.merge(df_cot, on=primary_keys, how='outer')
        else:
            merged_df = df_cot.copy()
    else:
        print(f"  경고: cot 파일이 없습니다.")
    
    # 3. cove 파일 처리
    if 'cove' in prompt_files:
        cove_file = prompt_files['cove']
        df_cove = pd.read_excel(cove_file)
        print(f"  cove 파일: {len(df_cove)} 행")
        
        # 기본키와 final_ans 컬럼만 선택
        columns_to_keep = primary_keys + ['final_ans']
        available_columns = [col for col in columns_to_keep if col in df_cove.columns]
        df_cove = df_cove[available_columns]
        
        # final_ans를 cove_ans로 이름 변경
        if 'final_ans' in df_cove.columns:
            df_cove = df_cove.rename(columns={'final_ans': 'cove_ans'})
        
        # 병합
        if merged_df is not None:
            merged_df = merged_df.merge(df_cove, on=primary_keys, how='outer')
        else:
            merged_df = df_cove.copy()
    else:
        print(f"  경고: cove 파일이 없습니다.")
    
    if merged_df is None:
        print(f"  오류: 병합할 파일이 없습니다.")
        return None
    
    # 컬럼 순서 정렬: 기본키, ques, label, ao_ans, cot_ans, cove_ans
    desired_order = primary_keys + ['ques', 'label', 'ao_ans', 'cot_ans', 'cove_ans']
    final_columns = [col for col in desired_order if col in merged_df.columns]
    
    # 누락된 컬럼이 있다면 추가 (뒤에 붙임)
    remaining_columns = [col for col in merged_df.columns if col not in final_columns]
    final_columns += remaining_columns
    
    merged_df = merged_df[final_columns]
    
    # 기본키로 정렬
    merged_df = merged_df.sort_values(by=primary_keys, ignore_index=True)
    
    print(f"  병합 완료: {len(merged_df)} 행, {len(merged_df.columns)} 컬럼")
    print(f"  컬럼: {', '.join(merged_df.columns)}")
    
    return merged_df


In [ ]:
# 모든 그룹에 대해 병합 수행
success_count = 0
failed_count = 0

for (model_name, question_type), prompt_files in sorted(file_groups.items()):
    try:
        merged_df = merge_prompt_files(model_name, question_type, prompt_files)
        
        if merged_df is not None:
            # 파일 저장
            output_filename = f"{model_name}_{question_type}_merged.xlsx"
            output_file = output_path / output_filename
            merged_df.to_excel(output_file, index=False)
            print(f"  저장 완료: {output_filename}")
            success_count += 1
        else:
            failed_count += 1
    except Exception as e:
        print(f"  오류 발생: {e}")
        failed_count += 1

print(f"\n\n===== 병합 완료 =====")
print(f"성공: {success_count}개")
print(f"실패: {failed_count}개")
print(f"\n병합된 파일들이 {output_path.absolute()} 폴더에 저장되었습니다.")


In [ ]:
# 결과 확인 (샘플)
output_files = list(output_path.glob('*.xlsx'))
if output_files:
    print(f"\n생성된 파일 목록:")
    for f in sorted(output_files):
        print(f"  - {f.name}")
    
    # 첫 번째 파일 샘플 출력
    sample_file = output_files[0]
    df_sample = pd.read_excel(sample_file)
    print(f"\n샘플 파일: {sample_file.name}")
    print(f"행 수: {len(df_sample)}, 컬럼 수: {len(df_sample.columns)}")
    print(f"\n컬럼: {list(df_sample.columns)}")
    print(f"\n상위 5행:")
    display(df_sample.head())
